# State-Space Trajectories of fMRI Activity

**Goal:** Compute PCA on a preprocessed naturalistic fMRI dataset (movie watching) and
visualise the resulting brain-state trajectories in low-dimensional PC space.

Two analysis levels:
1. **Whole-brain** — ROI-averaged signals (Schaefer 400-ROI atlas), shape `(n_rois, T)`
2. **Within-ROI** — voxel-by-time data inside a chosen ROI, shape `(n_voxels, T)`

Trajectory plots produced:
- PC1 vs PC2
- PC2 vs PC3
- PC1 vs PC3
- 3-D plot (PC1, PC2, PC3)

## 0 — Setup

In [ ]:
# Install dependencies (run once on Colab)
!pip install nilearn scikit-learn scipy matplotlib numpy nibabel tqdm --quiet

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, os

REPO_DIR = '/content/StateSpaceTrajectories'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone https://github.com/drgzkr/StateSpaceTrajectories.git {REPO_DIR} --quiet

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('neuro_tools ready at:', REPO_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from scipy.stats import zscore

# neuro_tools modules
from neuro_tools import atlas as nt_atlas
from neuro_tools import io as nt_io
from neuro_tools import decomposition as nt_dec
from neuro_tools import plotting as nt_plot
from neuro_tools import utils as nt_utils

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## 1 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/My Drive'
DATA_PATH    = f'{DRIVE_ROOT}/YourDataset/sub-01_task-movie_bold.nii.gz'

# Alternatively, load a pre-saved numpy array:
# DATA_PATH  = f'{DRIVE_ROOT}/YourDataset/whole_brain_data.npy'

# ── Atlas settings ─────────────────────────────────────────────────────────
N_ROIS           = 400
YEO_NETWORKS     = 17
ATLAS_RESOLUTION = 2   # mm

# ── PCA settings ───────────────────────────────────────────────────────────
N_COMPONENTS     = 10  # number of PCs to compute

# ── ROI for voxel-level analysis ───────────────────────────────────────────
# Schaefer label index (1-indexed).  Use nt_atlas labels list to identify.
ROI_IDX_FOR_VOXELWISE = 46   # example: change to the ROI you want

## 2 — Load Data

In [ ]:
import os

if DATA_PATH.endswith('.npy'):
    whole_brain_data = np.load(DATA_PATH, allow_pickle=True)
else:
    img_nifti        = nib.load(DATA_PATH)
    whole_brain_data = img_nifti.get_fdata()

print(f'Data shape: {whole_brain_data.shape}  →  (x, y, z, T)')

## 3 — Atlas & ROI Extraction

In [ ]:
atlas_resampled, roi_labels = nt_atlas.load_schaefer_atlas(
    reference_img  = DATA_PATH,
    n_rois         = N_ROIS,
    yeo_networks   = YEO_NETWORKS,
    resolution_mm  = ATLAS_RESOLUTION,
)
print(f'Atlas loaded: {N_ROIS} ROIs')
print('First 5 labels:', roi_labels[:5])

In [ ]:
from tqdm import trange

print('Computing ROI-averaged data matrix…')
roi_matrix = nt_io.compute_roi_averaged_matrix(
    whole_brain_data = whole_brain_data,
    atlas_resampled  = atlas_resampled,
    n_rois           = N_ROIS,
    zscore_output    = True,
)
print(f'ROI matrix shape: {roi_matrix.shape}  →  (n_rois, T)')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4))

im = axs[0].imshow(roi_matrix, aspect='auto', cmap='Spectral_r',
                   vmin=-3, vmax=3, interpolation='none')
axs[0].set_title('ROI × Time')
axs[0].set_xlabel('Time (TR)')
axs[0].set_ylabel('ROI index')
plt.colorbar(im, ax=axs[0], label='z-score')

tbt = nt_utils.time_by_time_correlation(roi_matrix)
im2 = axs[1].imshow(tbt, aspect='equal', cmap='RdBu_r',
                    vmin=-1, vmax=1, interpolation='none')
axs[1].set_title('Time × Time correlation')
axs[1].set_xlabel('Time (TR)')
axs[1].set_ylabel('Time (TR)')
plt.colorbar(im2, ax=axs[1], label='r')

fig.tight_layout()
plt.show()

## 4 — PCA: Whole-Brain ROI-Averaged Data

Each timepoint becomes a point in the N_ROIS-dimensional feature space.  
PCA rotates this space so the first axes capture maximum variance.

In [ ]:
scores_wb, components_wb, evr_wb, pca_wb = nt_dec.compute_pca(
    data         = roi_matrix,
    n_components = N_COMPONENTS,
)
print(f'PC scores shape: {scores_wb.shape}  →  (T, n_components)')
print('Variance explained by first 5 PCs:')
for i, v in enumerate(evr_wb[:5], 1):
    print(f'  PC{i}: {v*100:.2f}%')

In [ ]:
nt_plot.plot_explained_variance(evr_wb, n_show=N_COMPONENTS,
                                 title='Whole-brain PCA — Scree plot')
plt.show()

### 4.1 — Trajectory plots (whole brain)

In [ ]:
# Time index as colour (cool-warm gradient: early = blue, late = yellow)
time_color = np.arange(scores_wb.shape[0], dtype=float)

fig, axes = nt_plot.plot_trajectories_grid(
    scores        = scores_wb,
    pairs         = [(1, 2), (2, 3), (1, 3)],
    color_values  = time_color,
    cmap          = 'plasma',
    suptitle      = 'Whole-Brain State-Space Trajectories',
)
plt.show()

In [ ]:
fig, ax = nt_plot.plot_trajectory_3d(
    scores       = scores_wb,
    pc_x=1, pc_y=2, pc_z=3,
    color_values = time_color,
    cmap         = 'plasma',
    title        = 'Whole-Brain State-Space Trajectory (3D)',
)
plt.show()

### 4.2 — Most-traversed paths (whole brain)

Raw trajectories are noisy. The **density map** shows where the brain dwells most,
and the **flow field** overlays the dominant direction of traversal in each region
of PC space.

In [ ]:
# Dwell-time density maps — whole brain
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (px, py) in zip(axes, [(1, 2), (2, 3), (1, 3)]):
    nt_plot.plot_density_2d(scores_wb, pc_x=px, pc_y=py, ax=ax, cmap='YlOrRd')
fig.suptitle('Whole-Brain — Dwell-time Density', fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Flow field (density + velocity arrows) — whole brain
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (px, py) in zip(axes, [(1, 2), (2, 3), (1, 3)]):
    nt_plot.plot_flow_field(scores_wb, pc_x=px, pc_y=py, ax=ax,
                            n_bins=15, min_count=2, cmap='YlOrRd')
fig.suptitle('Whole-Brain — State-Space Flow Field', fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

### 4.3 — PC spatial maps (project loadings back to brain)

Each PC loading vector (one weight per ROI) is projected back into 3-D voxel
space via the atlas and then visualised on the fsaverage surface.

In [ ]:
for pc_num in [1, 2, 3]:
    loading_vec = components_wb[pc_num - 1, :]   # (n_rois,)

    pc_nifti = nt_io.roi_pattern_to_nifti(
        pattern         = loading_vec,
        atlas_resampled = atlas_resampled,
    )
    texture_l, texture_r = nt_plot.nifti_to_surface(pc_nifti)

    vabs = np.nanpercentile(np.abs(loading_vec), 95)
    nt_plot.long_plot(
        texture_l, texture_r,
        title    = f'PC {pc_num} spatial map (whole brain)',
        min_val  = -vabs, max_val = vabs,
        cmap     = 'RdBu_r',
        saving   = False,
    )

## 5 — PCA: Voxel-Level Data Within a Single ROI

Instead of averaging across voxels, we keep all voxels in one ROI and run PCA
on the `(n_voxels × T)` matrix.  The resulting trajectory reflects fine-grained
spatial patterns within the ROI.

In [ ]:
roi_label = roi_labels[ROI_IDX_FOR_VOXELWISE - 1]
print(f'Extracting voxel data for ROI {ROI_IDX_FOR_VOXELWISE}: {roi_label}')

roi_voxel_data = nt_io.get_roi_data(
    roi_idx         = ROI_IDX_FOR_VOXELWISE,
    whole_brain_data = whole_brain_data,
    atlas_resampled = atlas_resampled,
    clean           = True,
)
roi_voxel_data = zscore(roi_voxel_data, axis=-1)
print(f'Voxel data shape: {roi_voxel_data.shape}  →  (n_voxels, T)')

In [ ]:
scores_roi, components_roi, evr_roi, pca_roi = nt_dec.compute_pca(
    data         = roi_voxel_data,
    n_components = N_COMPONENTS,
)
print(f'PC scores shape: {scores_roi.shape}  →  (T, n_components)')
print(f'Variance explained by first 5 PCs:')
for i, v in enumerate(evr_roi[:5], 1):
    print(f'  PC{i}: {v*100:.2f}%')

In [ ]:
nt_plot.plot_explained_variance(
    evr_roi, n_show=N_COMPONENTS,
    title=f'Within-ROI PCA ({roi_label}) — Scree plot',
)
plt.show()

### 5.1 — Trajectory plots (within ROI)

In [ ]:
fig, axes = nt_plot.plot_trajectories_grid(
    scores        = scores_roi,
    pairs         = [(1, 2), (2, 3), (1, 3)],
    color_values  = time_color,
    cmap          = 'viridis',
    suptitle      = f'Within-ROI State-Space Trajectories\n({roi_label})',
)
plt.show()

In [ ]:
fig, ax = nt_plot.plot_trajectory_3d(
    scores       = scores_roi,
    pc_x=1, pc_y=2, pc_z=3,
    color_values = time_color,
    cmap         = 'viridis',
    title        = f'Within-ROI Trajectory (3D)\n{roi_label}',
)
plt.show()

### 5.2 — Most-traversed paths (within ROI)

In [ ]:
# Dwell-time density maps — within ROI
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (px, py) in zip(axes, [(1, 2), (2, 3), (1, 3)]):
    nt_plot.plot_density_2d(scores_roi, pc_x=px, pc_y=py, ax=ax, cmap='YlOrRd')
fig.suptitle(f'Within-ROI — Dwell-time Density\n({roi_label})', fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Flow field (density + velocity arrows) — within ROI
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (px, py) in zip(axes, [(1, 2), (2, 3), (1, 3)]):
    nt_plot.plot_flow_field(scores_roi, pc_x=px, pc_y=py, ax=ax,
                            n_bins=15, min_count=2, cmap='YlOrRd')
fig.suptitle(f'Within-ROI — State-Space Flow Field\n({roi_label})', fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

## 6 — Comparison: Whole-Brain vs Within-ROI

Overlay both PC1 trajectories on the same axis to compare their dynamics.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axs[0].plot(scores_wb[:, 0], color='steelblue', lw=1.2, label='Whole-brain PC1')
axs[0].axhline(0, color='k', lw=0.5, alpha=0.4)
axs[0].set_ylabel('Score')
axs[0].legend(frameon=False)
axs[0].spines[['top', 'right']].set_visible(False)

axs[1].plot(scores_roi[:, 0], color='darkorange', lw=1.2, label=f'{roi_label} PC1')
axs[1].axhline(0, color='k', lw=0.5, alpha=0.4)
axs[1].set_ylabel('Score')
axs[1].set_xlabel('Time (TR)')
axs[1].legend(frameon=False)
axs[1].spines[['top', 'right']].set_visible(False)

fig.suptitle('PC1 timeseries comparison', fontsize=12)
fig.tight_layout()
plt.show()

## 7 — Optional: Multiple ROIs Side-by-Side

Uncomment and adapt to compare trajectory geometry across several ROIs.

In [ ]:
# roi_indices_to_compare = [10, 46, 120, 250]  # pick ROIs of interest

# fig, axes = plt.subplots(2, len(roi_indices_to_compare),
#                          figsize=(5 * len(roi_indices_to_compare), 8))

# for col, roi_idx in enumerate(roi_indices_to_compare):
#     lbl   = roi_labels[roi_idx - 1]
#     vdata = nt_io.get_roi_data(roi_idx, whole_brain_data, atlas_resampled)
#     vdata = zscore(vdata, axis=-1)
#     sc, _, evr, _ = nt_dec.compute_pca(vdata, n_components=5)

#     # PC1 v PC2
#     nt_plot.plot_trajectory_2d(sc, pc_x=1, pc_y=2,
#                                 color_values=time_color, cmap='plasma',
#                                 title=lbl, ax=axes[0, col],
#                                 show_colorbar=(col == len(roi_indices_to_compare)-1))
#     # PC2 v PC3
#     nt_plot.plot_trajectory_2d(sc, pc_x=2, pc_y=3,
#                                 color_values=time_color, cmap='plasma',
#                                 ax=axes[1, col],
#                                 show_colorbar=(col == len(roi_indices_to_compare)-1))

# fig.suptitle('Multi-ROI Trajectory Comparison', fontsize=13)
# fig.tight_layout()
# plt.show()